# GhostNet-AI quickstart: side-scan sonar debris detection in two minutes

This notebook clones the repository, installs the AI package, gets the trained
weights, cuts the bundled demo survey (`demo/NBP0505_line01B_demo.xtf`, 40 MB,
a real Edgetech line) into 640 px tiles, and draws what the detector finds.

**No GPU needed.** On a CPU runtime the package picks ONNX Runtime when verified
ONNX exports are present (`GHOSTNET_RUNTIME=auto`). On this project's laptop,
detector + net U-Net took 0.44 s per tile with ONNX on CPU, 0.61 s with PyTorch
on CPU and 0.065 s with PyTorch on an RTX 3050 (`ai/experiments/onnx-cpu/RESULTS.md`).
Colab's free CPUs are slower than that laptop.

What the output is, and is not: every box is a **review candidate for a human
surveyor**, not a confirmed find. `ghost_net` is always `review_only`. See
`docs/READING_RESULTS.md`.

## 1. Settings

`WEIGHTS_URL` stays empty until the owner publishes the weights (`docs/PUBLISHING_WEIGHTS.md`).

In [ ]:
REPO = "https://github.com/vandit-cloud/ghostnet-ai.git"
BRANCH = "main"           # must contain the ONNX runtime code (branch feat/onnx-cpu until it is merged)
# Base URL the trained weights were published under, e.g.
#   https://huggingface.co/<org>/ghostnet-ai/resolve/main
#   https://github.com/<org>/ghostnet-ai/releases/download/<tag>
WEIGHTS_URL = ""
MAX_PINGS = None          # e.g. 3000 for a quicker look at part of the line

## 2. Clone and install

In [ ]:
import os, sys
if not os.path.exists("ghostnet-ai"):
    !git clone --depth 1 -b {BRANCH} {REPO} ghostnet-ai
%cd ghostnet-ai
# [onnx] adds onnxruntime for the CPU path. Colab already ships torch, so this
# mostly installs ultralytics and segmentation-models-pytorch.
!pip install -q -e "ai[onnx]"
sys.path.insert(0, "ai")

## 3. Weights

Each file is checked against the sha256 in its sidecar before it is installed;
a sidecar committed to git (e.g. `ai/models/trained/ghostnet.json`) is the
anchor, so a swapped download is refused. Without a URL, whatever the clone
already carries is used, and the package degrades rather than failing: no net
U-Net means `ghost_net` candidates come from the box detector.

In [ ]:
if WEIGHTS_URL:
    !python ai/scripts/fetch_weights.py --trained --url {WEIGHTS_URL} --with-onnx
else:
    print("WEIGHTS_URL not set; using the weights already in the clone:")
!ls -la ai/models/trained/

## 4. Detect on the demo survey

In [ ]:
import time
from pathlib import Path
from ghostnet import detect_survey
from ghostnet.config import Settings

settings = Settings()      # GHOSTNET_RUNTIME=auto: torch on a GPU, else verified ONNX, else torch CPU
print("runtime:", settings.provenance()["runtime"],
      "| net runtime:", settings.provenance().get("net_runtime", "no net model"))
if getattr(settings, "runtime_note", None):
    print("note:", settings.runtime_note)

t0 = time.perf_counter()
survey = detect_survey("demo/NBP0505_line01B_demo.xtf", "quickstart_tiles",
                       max_pings=MAX_PINGS, settings=settings)
dt = time.perf_counter() - t0
frames = survey["frames"]
n_det = sum(len(f["detections"]) for f in frames)
print(f"{len(frames)} tiles in {dt:.1f} s ({dt / max(len(frames), 1):.2f} s/tile, first-load included); "
      f"{n_det} detection(s)")
for w in survey["warnings"]:
    print("survey warning:", w)

## 5. Show the boxes

Tiles with detections first; if the line has none, the first tiles are shown so you can see what the sonar looks like.

In [ ]:
import cv2
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle, Polygon

COLOURS = {"debris": "tab:orange", "ghost_net": "tab:red", "ghost_pot": "tab:purple", "natural": "tab:green"}
hits = [f for f in frames if f["detections"]]
show = (hits + [f for f in frames if not f["detections"]])[:6]
fig, axes = plt.subplots(1, len(show), figsize=(4 * len(show), 4.4), squeeze=False)
for ax, f in zip(axes[0], show):
    img = cv2.imread(str(Path("quickstart_tiles") / (f["frame_id"] + ".png")), cv2.IMREAD_GRAYSCALE)
    if img is None:     # tile naming fallback
        img = cv2.imread(str(next(Path("quickstart_tiles").glob(f["frame_id"] + ".*"))), cv2.IMREAD_GRAYSCALE)
    ax.imshow(img, cmap="copper")
    for d in f["detections"]:
        x, y, w, h = d["bbox"]
        c = COLOURS.get(d["class"], "cyan")
        ax.add_patch(Rectangle((x, y), w, h, fill=False, lw=2, ec=c))
        if d.get("mask"):
            ax.add_patch(Polygon(d["mask"], closed=True, fill=False, lw=1, ec=c, ls="--"))
        ax.text(x, y - 4, f'{d["class"]} {d["calibrated_confidence"]:.2f}', color=c, fontsize=8)
    ax.set_title(f["frame_id"][-24:], fontsize=8)
    ax.axis("off")
plt.tight_layout()
plt.show()

In [ ]:
import json
# One full payload, as the web app receives it (contracts/ai-output.schema.json).
print(json.dumps((hits or frames)[0], indent=1)[:3000])

## Next

* `GHOSTNET_RUNTIME=torch|onnx|auto` picks the executor; provenance in every payload says which ran.
* `python ai/scripts/export_onnx.py --images <>=50 real tiles>` builds and parity-checks ONNX exports yourself.
* The full web app (map, review queue, reports): `docs/SETUP.md`.
* Licences: code Apache-2.0; our annotations CC BY 4.0; third-party imagery keeps its own terms (`ATTRIBUTION.md`).